# Task B — Smoothing, windows, and framing (computed)

This notebook repeats Part D's documented hourly-to-daily cleaning so it
can be run independently. The observed hourly series keeps gaps as `NaN`.
For a comparable daily-flow estimate, missing hours are imputed with the
median for the same weekday and hour; raw partial-day sums stay separate
and the imputation count is reported.

**Data:** synthetic `sessions_hourly.csv`. The daily unit is
**sessions per day (sum of hourly flow estimates)**. Observed partial-day
sums are kept separately; missing hourly bins are imputed by weekday-hour
median only for the trend/decomposition estimate.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd()
WEEK4_DIR = HERE if (HERE / "assignments" / "data").exists() else HERE / "week4"
ASSIGNMENTS_DIR = WEEK4_DIR / "assignments"
DATA_DIR = ASSIGNMENTS_DIR / "data"
sys.path.insert(0, str(ASSIGNMENTS_DIR))
from tslib import PALETTE, style_defaults, save_fig, audit_index

style_defaults()

from tslib import window_report, lag_at_turn
from statsmodels.tsa.seasonal import seasonal_decompose

raw = pd.read_csv(DATA_DIR / "sessions_hourly.csv")
iso = raw["ts"].str.match(r"^\d{4}-\d{2}-\d{2} ")
parsed = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")
parsed.loc[iso] = pd.to_datetime(raw.loc[iso, "ts"], format="%Y-%m-%d %H:%M:%S")
parsed.loc[~iso] = pd.to_datetime(raw.loc[~iso, "ts"], format="%d/%m/%Y %H:%M")
assert parsed.notna().all(), "Unexpected timestamp format; inspect before continuing."

work = raw.assign(ts=parsed, device=raw["device"].str.strip().str.title())
bad_sentinel = work["sessions"].lt(0)
deduped = work.loc[~bad_sentinel].drop_duplicates("ts", keep="first")
deduped = deduped.set_index("ts").sort_index()
hours = pd.date_range(deduped.index.min().normalize(),
                      deduped.index.max().normalize() + pd.Timedelta(hours=23), freq="h")
hourly = deduped["sessions"].reindex(hours).astype(float)
coverage = hourly.notna().resample("D").sum()
profile = deduped["sessions"].groupby([deduped.index.dayofweek, deduped.index.hour]).median()
hourly_estimate = hourly.copy()
missing_hours = hourly_estimate.index[hourly_estimate.isna()]
missing_keys = pd.MultiIndex.from_arrays([missing_hours.dayofweek, missing_hours.hour])
hourly_estimate.loc[missing_hours] = [profile.loc[key] for key in missing_keys]
observed_daily = hourly.resample("D").sum(min_count=1).asfreq("D")
daily = hourly_estimate.resample("D").sum(min_count=24).asfreq("D")
daily.name = "sessions_per_day"
print(f"Input rows={len(raw):,}; sentinel rows removed={int(bad_sentinel.sum()):,}; "
      f"duplicate timestamps removed={len(work.loc[~bad_sentinel]) - len(deduped):,}")
print(f"Missing hourly bins imputed for daily estimates={len(missing_hours):,}; "
      f"days with incomplete observed coverage={int(coverage.lt(24).sum()):,}; "
      f"dates={daily.index.min().date()} to {daily.index.max().date()}")

Input rows=12,670; sentinel rows removed=290; duplicate timestamps removed=283
Missing hourly bins imputed for daily estimates=599; days with incomplete observed coverage=235; dates=2024-04-01 to 2025-09-11


## 1. What each centred window removes

In [2]:
report = window_report(daily, windows=(7, 28, 91), center=True)
display(report)
print("Selection for the retrospective Part D review: 7 days, fixed before plotting, "
      "because it targets the weekly rhythm. It is not selected for looking smooth.")

,window,raw_sd,smoothed_sd,variation_removed_pct,periods_lost_each_end
0,7,6728.01,4545.2302,32.4,3
1,28,6728.01,4324.7629,35.7,14
2,91,6728.01,4038.8178,40.0,45


Selection for the retrospective Part D review: 7 days, fixed before plotting, because it targets the weekly rhythm. It is not selected for looking smooth.


`variation_removed_pct` is the change in standard deviation of the smoothed
series, not proof that every removed movement was noise. A seven-day window
cancels much of the day-of-week cycle; 28 days suppress the weekly rhythm
and month-scale variation; 91 days suppress weekly and month-scale
variation in favour of quarter-scale movement. These are low-pass windows,
not exact seasonal filters. The edge counts in the report are periods lost
at **each** end. The 7-day choice is retrospective and centred; it is not
suitable for a live alert.

## 2. Three centred smoothers at the Black Friday spike

In [3]:
target = pd.Timestamp("2024-11-29")
rolling14 = daily.rolling(14, center=True, min_periods=14)
mean_at = rolling14.mean().loc[target]
median_at = rolling14.median().loc[target]
alpha = 2 / (14 + 1)
def centered_exp(values):
    offsets = np.arange(len(values), dtype=float) - (len(values) - 1) / 2
    weights = (1 - alpha) ** np.abs(offsets)
    return float(np.average(values, weights=weights))
ewma_at = rolling14.apply(centered_exp, raw=True).loc[target]
raw_at = observed_daily.loc[target]
estimate_at = daily.loc[target]
print(f"2024-11-29 observed sum of available hours: {raw_at:,.0f}")
print(f"2024-11-29 weekday-hour-imputed daily estimate: {estimate_at:,.0f}")
print(f"14-day centred mean: {mean_at:,.1f}")
print(f"14-day centred median: {median_at:,.1f}")
print(f"14-day centred exponential kernel (alpha={alpha:.4f}): {ewma_at:,.1f}")
print("Monitoring: use a trailing EWMA and caption it as trailing/no-look-ahead. "
      "Quarterly review: use the centred median (and keep the raw spike visible) "
      "when a robust summary is appropriate.")

2024-11-29 observed sum of available hours: 113,710
2024-11-29 weekday-hour-imputed daily estimate: 113,710
14-day centred mean: 33,402.7
14-day centred median: 24,765.5
14-day centred exponential kernel (alpha=0.1333): 36,441.8
Monitoring: use a trailing EWMA and caption it as trailing/no-look-ahead. Quarterly review: use the centred median (and keep the raw spike visible) when a robust summary is appropriate.


The centred exponential smoother uses symmetric weights
`(1 - alpha) ** abs(day_offset)`, with `alpha = 2 / (14 + 1)`. It is
centred—not a live one-sided EWMA—and is stated this way to make its kernel
reproducible. Keep the raw observation on the chart: smoothing does not
delete the Black Friday event.

## 3. Weekly decomposition and the annual-cycle limitation

In [4]:
print(f"Decomposition uses daily estimates with {len(missing_hours)} weekday-hour median imputations; "
      "these are model inputs, not observed hourly traffic.")
weekly = seasonal_decompose(daily, model="additive", period=7,
                            extrapolate_trend=0)
weekly_amplitude = weekly.seasonal.max() - weekly.seasonal.min()
largest_residual_date = weekly.resid.abs().idxmax()
print(f"Weekly seasonal peak-to-trough amplitude: {weekly_amplitude:,.1f} sessions/day")
print(f"Largest absolute weekly-decomposition residual: {largest_residual_date.date()} "
      f"({weekly.resid.loc[largest_residual_date]:,.1f} sessions/day)")

try:
    annual = seasonal_decompose(daily, model="additive", period=365,
                                extrapolate_trend=0)
    print("Annual component estimated; compare seasonal amplitude and residuals before interpretation.")
except ValueError as exc:
    print("Annual period=365 could not be estimated:", exc)
    print(f"Available days={len(daily)}; the classical additive decomposition "
          "requires at least two complete periods (730 days). No annual seasonal/residual "
          "reallocation is defensible from this extract.")

Decomposition uses daily estimates with 599 weekday-hour median imputations; these are model inputs, not observed hourly traffic.
Weekly seasonal peak-to-trough amplitude: 7,678.5 sessions/day
Largest absolute weekly-decomposition residual: 2024-11-29 (74,830.1 sessions/day)
Annual period=365 could not be estimated: x must have 2 complete cycles requires 730 observations. x only has 529 observation(s)
Available days=529; the classical additive decomposition requires at least two complete periods (730 days). No annual seasonal/residual reallocation is defensible from this extract.


**Interpretation.** The weekly decomposition is explicitly conditional on
weekday-hour median imputations for absent/sentinel hourly records; those
values are identified above. This
extract contains fewer than two full 365-day cycles, so the classical
`seasonal_decompose(period=365)` cannot identify an annual seasonal
component. Do not claim that an annual effect moved from residual to
seasonal when the method's minimum evidence requirement is not met.

## 4. Trailing versus centred 31-day mean

In [5]:
turn = lag_at_turn(daily, window=31, how="min")
print(turn)
print(f"Measured lag={turn['lag_periods']} days; half-window={turn['half_window']} days.")
print("A dashboard refreshed each morning must use a trailing window, since future observations "
      "do not exist yet; caption it '31-day trailing mean, latest value uses prior 30 days'.")

{'window': 31, 'trailing_turn': Timestamp('2024-05-16 00:00:00'), 'centred_turn': Timestamp('2024-05-01 00:00:00'), 'lag_periods': 15, 'half_window': 15}
Measured lag=15 days; half-window=15 days.
A dashboard refreshed each morning must use a trailing window, since future observations do not exist yet; caption it '31-day trailing mean, latest value uses prior 30 days'.


## 5. Count how often the framing changes trend direction

In [6]:
window_lengths = (7, 14, 28, 56, 91)
trend_rows = []
for month in range(1, 13):
    candidates = daily.index[daily.index.month == month]
    if len(candidates) == 0:
        continue
    start = candidates[0]
    for window in window_lengths:
        smoothed = daily.rolling(window, center=True, min_periods=window).mean()
        segment = smoothed.loc[start:].dropna()
        slope = np.polyfit(np.arange(len(segment)), segment.to_numpy(), 1)[0]
        trend_rows.append({"start_month": month, "start_date": start.date(),
                           "window_days": window, "slope_sessions_per_day": slope,
                           "direction": "rising" if slope > 0 else "falling"})
trend_counts = pd.Series([r["direction"] for r in trend_rows]).value_counts()
print("All 12 first-occurring calendar-month starts × 5 centred windows:")
display(pd.DataFrame(trend_rows))
print("Rising combinations:", int(trend_counts.get("rising", 0)),
      "| falling combinations:", int(trend_counts.get("falling", 0)),
      "| total:", len(trend_rows))

All 12 first-occurring calendar-month starts × 5 centred windows:


,start_month,start_date,window_days,slope_sessions_per_day,direction
0,1,2025-01-01,7,6.414377,rising
1,1,2025-01-01,14,4.788995,rising
2,1,2025-01-01,28,1.830002,rising
3,1,2025-01-01,56,-3.723888,falling
4,1,2025-01-01,91,-10.529328,falling
5,2,2025-02-01,7,16.657184,rising
6,2,2025-02-01,14,14.838786,rising
7,2,2025-02-01,28,11.495518,rising
8,2,2025-02-01,56,4.999840,rising
9,2,2025-02-01,91,-1.603464,falling


Rising combinations: 39 | falling combinations: 21 | total: 60


The Lie Factor is undefined for a cherry-picked time window: it requires a
reference effect in the data and a corresponding effect encoded by the
graphic, while the chosen start/window has no neutral, prespecified
reference and changes which data are considered. The honest substitute is
to declare the window and start date before plotting, show the complete
available series, and report a sensitivity analysis across the stated
alternatives (as above)—not to invent a Lie Factor.

## Reproducibility note

Part D uses the same deterministic parser, duplicate/sentinel rule, hourly
calendar, and 24-valid-hours daily aggregation. The final September 2025
extract is partial and is marked rather than compared with a complete
month. No missing day is silently interpolated into the displayed raw
series.